In [ ]:
# Run this in 01_data_exploration.ipynb — Cell 1
import sys, shutil, os
sys.path.append("..")

# Clean old indexes
for path in [
    "../data/processed/chunks.json",
    "../data/processed/manifest.json",
    "../vector_rag/chroma_db",
    "../vectorless_rag/bm25_index.pkl",
    "../vectorless_rag/bm25_manifest.json"
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"🗑️  {path}")
    elif os.path.exists(path):
        os.remove(path)
        print(f"🗑️  {path}")

print("\n✅ Clean — ready to rebuild")

In [ ]:
# Cell 2 — Rebuild with new architecture
from data_loader             import run_preprocessing_pipeline
from vector_rag.indexer      import index_chunks
from vectorless_rag.indexer  import build_bm25_index

data = run_preprocessing_pipeline()

print(f"Parents  : {len(data['parents'])}")
print(f"Children : {len(data['children'])}")

index_chunks(data)
build_bm25_index(data)

print("\n🎉 All indexes rebuilt with improved architecture!")

In [ ]:
# Cell 3 — Quick retrieval test
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

q = "What was NVIDIA's total revenue?"

r1 = vec.ask(q)
r2 = vl.ask(q)

vec.show(r1)
vl.show(r2)

In [ ]:
import traceback
from vectorless_rag.pipeline import VectorlessRAGPipeline

try:
    vl = VectorlessRAGPipeline()
    r2 = vl.ask("What was NVIDIA's total revenue?")
    print("Answer:", r2)
    vl.show(r2)
except Exception as e:
    print("FULL ERROR:")
    traceback.print_exc()
    print("\nException details:", str(e))

In [2]:
import sys, os, shutil
sys.path.append("..")

stale = [
    "../vector_rag/chroma_db",
    "../vectorless_rag/bm25_index.pkl",
    "../vectorless_rag/bm25_manifest.json",
    "../data/processed/chunks.json",
    "../data/processed/manifest.json",
]
for path in stale:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"🗑️  Deleted dir:  {path}")
    elif os.path.exists(path):
        os.remove(path)
        print(f"🗑️  Deleted file: {path}")
    else:
        print(f"✅  Not present:  {path}")

✅  Not present:  ../vector_rag/chroma_db
✅  Not present:  ../vectorless_rag/bm25_index.pkl
✅  Not present:  ../vectorless_rag/bm25_manifest.json
✅  Not present:  ../data/processed/chunks.json
✅  Not present:  ../data/processed/manifest.json


In [3]:
from data_loader import run_preprocessing_pipeline
from vector_rag.indexer import index_chunks
from vectorless_rag.indexer import build_bm25_index

data = run_preprocessing_pipeline()
print(f"Parents: {len(data['parents'])} | Children: {len(data['children'])}")

index_chunks(data)      # BGE model downloads ~438MB on first run — wait for it
build_bm25_index(data)
print("🎉 Done")

   PHASE 2 — SCALABLE PREPROCESSING PIPELINE
  🆕 New PDF detected: amazon_10k.pdf
  🆕 New PDF detected: microsoft_10k.pdf
  🆕 New PDF detected: netflix_10k.pdf
  🆕 New PDF detected: nvidia_10k.pdf

📂 Processing 4 new PDF(s)...



Processing PDFs:  25%|██▌       | 1/4 [00:00<00:01,  2.22it/s]

   ✅ AMAZON_10K: 90 pages → 410 parents, 1468 children


Processing PDFs:  50%|█████     | 2/4 [00:03<00:04,  2.02s/it]

   ✅ MICROSOFT_10K: 156 pages → 629 parents, 2242 children


...

   ✅ NVIDIA_10K: 93 pages → 435 parents, 1914 children

💾 Saved 2010 parents, 7913 children
📋 Manifest: 4 PDFs tracked

🎉 Preprocessing complete!

Parents: 2010 | Children: 7913
   VECTOR RAG INDEXER — Parent-Child + BGE + HNSW


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📤 Indexing 7913 child chunks...
   Embedding model: BAAI/bge-base-en-v1.5


Embedding children: 100%|██████████| 124/124 [09:38<00:00,  4.67s/it]


✅ ChromaDB updated — 2289 total vectors

   VECTORLESS RAG INDEXER — BM25 + Financial Tokenizer

✅ BM25 already up to date — 7913 children indexed

🎉 Done


In [4]:
from vector_rag.pipeline import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline
from hybrid_rag.pipeline import HybridRAGPipeline

vec    = VectorRAGPipeline()
vl     = VectorlessRAGPipeline()
hybrid = HybridRAGPipeline()

q = "What was Amazon Web Services revenue for the most recent fiscal year?"
print("[VECTOR]    ", vec.ask(q)["answer"][:200])
print("[VECTORLESS]", vl.ask(q)["answer"][:200])
print("[HYBRID]    ", hybrid.ask(q)["answer"][:200])

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 2289 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

🔧 Initialising Hybrid RAG Pipeline...
✅ ChromaDB loaded — 2289 child vectors
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Hybrid RAG ready — 2289 vectors | 7913 BM25 children | 2010 parents

🔁 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready
[VECTOR]     This information was not found in the retrieved sections of **NVIDIA's** 10-K report. The context does not disclose revenue figures for **Amazon Web Services (AWS)** or any other specific customer by 
[VECTORLESS] For **Amazon**, AWS (Amazon Web Services) revenue in the most recent fiscal year (2025) was **$129 billion**, reflecting a **20% year-over-year increase** from $108 billion in 2024. This figure is sou
[HYBRID]     For **Amazon**, the AWS (Amazon Web Services) revenue for the most recent fiscal year (2025) was **$129 billion**, reflecting a **20% year-over-year increase** from $108 billion in 2024. This figure i
